# Inhouse TTS · entrenamiento del modelo propio (IHV1) en Google Colab

Este cuaderno entrena **nuestro propio modelo de voz**, desde cero y sin modelos de terceros, en la GPU de Colab. Todo el código que entrena es **Rust** (framework *burn* con CUDA). El cuaderno solo lanza los comandos.

**Cómo usarlo**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU*** (T4 en la versión gratuita; L4 o A100 en Colab Pro).
2. *Entorno de ejecución → **Ejecutar todo***. La primera vez, Colab pide permiso para usar tu Google Drive.
3. Si la sesión se corta, vuelve a abrir el cuaderno y pulsa *Ejecutar todo* otra vez. Los datos y los checkpoints quedan en `MyDrive/inhouse-tts`, y el entrenamiento **continúa donde se quedó**.

| Paso | Primera vez | Siguientes sesiones |
|---|---|---|
| Entrenador en Rust | ~1 min (binario precompilado) o ~20 min (compilación) | ~1 min |
| Descargar y preparar los datos | ~20–40 min | ~2 min (copia desde Drive) |
| Entrenar | las horas que dure la sesión | continúa |

`MyDrive/inhouse-tts/runs/<voz>/samples/` va guardando muestras de audio para escuchar el progreso.

In [ ]:
#@title Configuración
VOICE = "es-ES"   #@param ["es-ES"]
PRESET = "small"  #@param ["small", "base", "tiny"]
STEPS = 300000    #@param {type:"integer"}
REPO = "https://github.com/miguelcoxcaballero/inhouse-TTS"
import os
WORK = "/content/drive/MyDrive/inhouse-tts"
os.environ.update(VOICE=VOICE, PRESET=PRESET, STEPS=str(STEPS), REPO=REPO, WORK=WORK)

In [ ]:
#@title 1 · GPU y Google Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "$WORK/data" "$WORK/runs"

### 2 · Entrenador en Rust: binario precompilado (CUDA) o compilación desde el código

In [ ]:
%%bash
set -e
cd /content
URL="$REPO/releases/download/trainer-latest/ihtts-trainer-cuda-linux-x86_64.tar.gz"
if curl -fsSL "$URL" -o trainer.tgz && tar -xzf trainer.tgz; then
  REV=$(cat ihtts/REVISION)
  echo "binario precompilado de $REV"
else
  REV=main
fi
test -d inhouse-tts || git clone -q "$REPO" inhouse-tts
cd inhouse-tts && git fetch -q origin && git checkout -q "$REV" && cd ..
mkdir -p bin
if [ -x ihtts/ihtts-train ]; then
  cp ihtts/ihtts-train ihtts/ihtts-data bin/
else
  test -x $HOME/.cargo/bin/cargo || (curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q)
  export PATH=$HOME/.cargo/bin:$PATH
  cd inhouse-tts
  cargo build --profile train -p inhouse-tts-data -p inhouse-tts-train --features inhouse-tts-train/cuda
  cp target/train/ihtts-train target/train/ihtts-data ../bin/
fi
/content/bin/ihtts-train 2>&1 | head -1

### 3 · Datos: descarga y preparación, con copia en Drive

In [ ]:
%%bash
set -e
cd /content/inhouse-tts
PACK="$WORK/data/$VOICE.tar"
mkdir -p /content/data
if [ -f "$PACK" ]; then
  tar -xf "$PACK" -C /content/data
else
  /content/bin/ihtts-data prepare corpora/$VOICE/*.json --cache /content/cache --out /content/data
  tar -cf "$PACK" -C /content/data .
  rm -rf /content/cache
fi
cat /content/data/*/report.json | grep -E '"(utterances|hours)"'

### 4 · Entrenar (continúa automáticamente desde el último checkpoint)

In [ ]:
%%bash
cd /content/inhouse-tts
DATA=$(ls /content/data/*/manifest.jsonl | sed 's/^/--data /' | tr '\n' ' ')
/content/bin/ihtts-train train --run "$WORK/runs/$VOICE" --preset "$PRESET" $DATA \
  --device cuda --steps "$STEPS" --resume 2>&1

In [ ]:
#@title 5 · Escuchar las últimas muestras
import glob, IPython.display as ipd
latest = sorted(glob.glob(f"{WORK}/runs/{VOICE}/samples/*"))[-1:]
for d in latest:
    print(d)
    for w in sorted(glob.glob(d + "/*.wav"))[:6]:
        print(os.path.basename(w)); ipd.display(ipd.Audio(w))

In [ ]:
#@title 6 · Exportar el modelo al formato de Inhouse TTS (.ihw)
!/content/bin/ihtts-train export --run "$WORK/runs/$VOICE" --out "$WORK/ihv1-$VOICE.ihw"
print("Modelo en Google Drive:", f"{WORK}/ihv1-{VOICE}.ihw")